In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/kaggle/input/customer-churn-prediction-dataset/customer_churn_dataset.csv')
df.head()

,customer_id,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn
0,1,52,54.20,2818.40,Month-to-month,Credit,DSL,No,Yes,1,No
1,2,15,35.28,529.20,Month-to-month,Debit,DSL,No,No,2,No
2,3,72,78.24,5633.28,Month-to-month,Debit,DSL,No,No,0,No
3,4,61,80.24,4894.64,One year,Cash,Fiber,Yes,Yes,0,No
4,5,21,39.38,826.98,Month-to-month,UPI,Fiber,No,No,4,Yes


In [3]:
df.columns

Index(['customer_id', 'tenure', 'monthly_charges', 'total_charges', 'contract',
       'payment_method', 'internet_service', 'tech_support', 'online_security',
       'support_calls', 'churn'],
      dtype='object')

In [4]:
#axis = 1 => we decided to drop the column
df1 = df.drop('customer_id' , axis = 1)
df1.head()

,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn
0,52,54.20,2818.40,Month-to-month,Credit,DSL,No,Yes,1,No
1,15,35.28,529.20,Month-to-month,Debit,DSL,No,No,2,No
2,72,78.24,5633.28,Month-to-month,Debit,DSL,No,No,0,No
3,61,80.24,4894.64,One year,Cash,Fiber,Yes,Yes,0,No
4,21,39.38,826.98,Month-to-month,UPI,Fiber,No,No,4,Yes


In [5]:
len(df1)

20000

In [6]:
df1.isna().sum()

tenure                 0
monthly_charges        0
total_charges          0
contract               0
payment_method         0
internet_service    2013
tech_support           0
online_security        0
support_calls          0
churn                  0
dtype: int64

In [7]:
df1['internet_service'].unique()

array(['DSL', 'Fiber', nan], dtype=object)

In [8]:
df1['internet_service'].value_counts()

internet_service
Fiber    10064
DSL       7923
Name: count, dtype: int64

In [9]:
#fill the missing values with 'Unknown'
df1['internet_service'] = df1['internet_service'].fillna('Unknown')

In [10]:
df1.isna().sum()

tenure              0
monthly_charges     0
total_charges       0
contract            0
payment_method      0
internet_service    0
tech_support        0
online_security     0
support_calls       0
churn               0
dtype: int64

In [11]:
df1.dtypes

tenure                int64
monthly_charges     float64
total_charges       float64
contract             object
payment_method       object
internet_service     object
tech_support         object
online_security      object
support_calls         int64
churn                object
dtype: object

In [12]:
print(df1['contract'].unique())
print(df1['payment_method'].unique())
print(df1['internet_service'].unique())
print(df1['tech_support'].unique())
print(df1['online_security'].unique())

['Month-to-month' 'One year' 'Two year']
['Credit' 'Debit' 'Cash' 'UPI']
['DSL' 'Fiber' 'Unknown']
['No' 'Yes']
['Yes' 'No']


**Data Encoding**

In [13]:
df2 = df1.copy()

In [14]:
#Integer Encoding or label encoding
df2['contract'] = df2['contract'].map({'Month-to-month' : 0,
                                       'One year' : 1,
                                       'Two year' : 2})
df2['tech_support'] = df2['tech_support'].map({'No':0, 'Yes':1})
df2['online_security'] = df2['online_security'].map({'No':0, 'Yes':1})

In [15]:
#One - hot encoding
pm = pd.get_dummies(df2['payment_method'] , dtype = int)
isr = pd.get_dummies(df2['internet_service'] , dtype = int)

In [16]:
pm

,Cash,Credit,Debit,UPI
0,0,1,0,0
1,0,0,1,0
2,0,0,1,0
3,1,0,0,0
4,0,0,0,1
...,...,...,...,...
19995,1,0,0,0
19996,0,0,0,1
19997,0,1,0,0
19998,0,0,1,0


In [17]:
df2.head()

,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn
0,52,54.20,2818.40,0,Credit,DSL,0,1,1,No
1,15,35.28,529.20,0,Debit,DSL,0,0,2,No
2,72,78.24,5633.28,0,Debit,DSL,0,0,0,No
3,61,80.24,4894.64,1,Cash,Fiber,1,1,0,No
4,21,39.38,826.98,0,UPI,Fiber,0,0,4,Yes


In [18]:
#pd.concat([df2 , pm] , axis =1)

In [19]:
df3 = pd.concat([df2, pm , isr] , axis = 1).drop(['payment_method',
                                                  'internet_service'],
                                                 axis = 1)

In [20]:
df3.head()

,tenure,monthly_charges,total_charges,contract,tech_support,online_security,support_calls,churn,Cash,Credit,Debit,UPI,DSL,Fiber,Unknown
0,52,54.20,2818.40,0,0,1,1,No,0,1,0,0,1,0,0
1,15,35.28,529.20,0,0,0,2,No,0,0,1,0,1,0,0
2,72,78.24,5633.28,0,0,0,0,No,0,0,1,0,1,0,0
3,61,80.24,4894.64,1,1,1,0,No,1,0,0,0,0,1,0
4,21,39.38,826.98,0,0,0,4,Yes,0,0,0,1,0,1,0


**Define features and labels - X & y**

In [21]:
X = df3.drop('churn' , axis = 1)
y = df3['churn']

In [22]:
y.value_counts()

churn
No     13157
Yes     6843
Name: count, dtype: int64

In [23]:
X.head()

,tenure,monthly_charges,total_charges,contract,tech_support,online_security,support_calls,Cash,Credit,Debit,UPI,DSL,Fiber,Unknown
0,52,54.20,2818.40,0,0,1,1,0,1,0,0,1,0,0
1,15,35.28,529.20,0,0,0,2,0,0,1,0,1,0,0
2,72,78.24,5633.28,0,0,0,0,0,0,1,0,1,0,0
3,61,80.24,4894.64,1,1,1,0,1,0,0,0,0,1,0
4,21,39.38,826.98,0,0,0,4,0,0,0,1,0,1,0


**Split the data into train & test**

In [24]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25)

In [25]:
print(X.shape)
print(xtrain.shape)
print(xtest.shape)

(20000, 14)
(15000, 14)
(5000, 14)


**Import Algorithm**

In [26]:
from sklearn.linear_model import LogisticRegression
model_log = LogisticRegression()
model_log.fit(xtrain, ytrain)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [27]:
pd.DataFrame(model_log.coef_ , columns = xtrain.columns)

,tenure,monthly_charges,total_charges,contract,tech_support,online_security,support_calls,Cash,Credit,Debit,UPI,DSL,Fiber,Unknown
0,-0.016713,0.018981,0.000075,-0.793738,-1.195751,0.071345,0.40217,-0.25261,-0.223383,-0.228314,-0.24538,-0.381079,-0.323293,-0.245315


In [28]:
#prediction made on training data
ytrainPred = model_log.predict(xtrain)

#prediction made on test data
ytestPred = model_log.predict(xtest)

In [29]:
#compare real labels with predicted outcomes
(ytrain == ytrainPred).sum()/len(xtrain)

np.float64(0.7707333333333334)

In [30]:
from sklearn.metrics import accuracy_score
print("Train data accuracy", accuracy_score(ytrain, ytrainPred))
print("Test data accuracy", accuracy_score(ytest, ytestPred))

Train data accuracy 0.7707333333333334
Test data accuracy 0.7744


**Confusion Matrix**

In [31]:
from sklearn.metrics import confusion_matrix, classification_report

In [32]:
print(confusion_matrix(ytrain, ytrainPred))

[[8866 1005]
 [2434 2695]]


In [35]:
print(classification_report(ytrain, ytrainPred))

              precision    recall  f1-score   support

          No       0.78      0.90      0.84      9871
         Yes       0.73      0.53      0.61      5129

    accuracy                           0.77     15000
   macro avg       0.76      0.71      0.72     15000
weighted avg       0.77      0.77      0.76     15000



In [33]:
print(confusion_matrix(ytest, ytestPred))

[[2961  325]
 [ 803  911]]


In [34]:
ytest.value_counts()

churn
No     3286
Yes    1714
Name: count, dtype: int64

In [36]:
print(classification_report(ytest, ytestPred))

              precision    recall  f1-score   support

          No       0.79      0.90      0.84      3286
         Yes       0.74      0.53      0.62      1714

    accuracy                           0.77      5000
   macro avg       0.76      0.72      0.73      5000
weighted avg       0.77      0.77      0.76      5000

